<a href="https://www.kaggle.com/code/antongalysh/pytorch?scriptVersionId=339726536" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Завдання

У вашому розпорядженні є дані про діаманти:

* **вага** (у каратах);
* **ціна** (у доларах).

Ваше завдання — навчити **дві нейронні мережі** прогнозувати ціну діаманта за його вагою та порівняти, як змінюється помилка під час навчання.

Для оцінки якості моделі використайте готову функцію втрат **Mean Squared Error (MSE)** з бібліотеки **PyTorch**.

```python
loss_fn = torch.nn.MSELoss()
```

### кроки

1. Створіть **дві** нейронні мережі:
   * `model1`;
   * `model2`.
    
2. Для кожної моделі створіть окремого вчителя:
   * `teacher1`;
   * `teacher2`.
     
3. Створіть два списки:
   * `losses1`;
   * `losses2`.

4. Створіть цикл, який виконається **100_000 разів**. Для зручності скористайтесь
```python
from tqdm import tqdm

for _ in tqdm(range(100_0000)):
    ...
```

5. На кожній ітерації для **кожної** моделі виконайте такі кроки:
   * отримайте прогноз моделі;
   * обчисліть помилку за допомогою `loss_fn`;
   * викличте метод `teach()` відповідного вчителя;
   * додайте значення помилки до відповідного списку (`losses1` або `losses2`).
    
6. Після завершення навчання:
   * отримайте прогноз для кожної моделі;
   * виведіть прогнозовані ціни;
   * виведіть фінальне значення помилки;
   * виведіть списки `losses1` та `losses2`.
     ```python
     import matplotlib.pyplot as plt
    
     plt.plot(losses1, label="model1")
     plt.plot(losses2, label="model2")
     plt.legend()
     ```
7. Порівняйте, як змінювалася помилка під час навчання двох моделей.

8. Спробуйте зменшити навчальний крок у вчителів(вибарайте між 0.01 0.001 ...)

9. спробуйте підключитись до графічного процесора та натренувати нейромережі(у вас виникне помилка, проаналізуйте її)


In [39]:
import pandas as pd

In [40]:
df = pd.read_csv("https://raw.githubusercontent.com/HalyshAnton/IT-Step-Pyton-AI/main/module3/data/diamonds.csv",
                 index_col=0)

df.head()

,carat,price
0,0.23,326
1,0.21,326
2,0.23,327
3,0.29,334
4,0.31,335


In [41]:
class NeuralNetwork1:
    def __init__(self, device="cpu"):
        self.device = device

        self.price_per_carat = torch.tensor(
            [50.],   # початкова ціна за карат
            requires_grad=True,
            device=device
        )

        self.other = torch.tensor(
            [42.],  # ціна за додаткові витрати
            requires_grad=True,
            device=device
        )

    def predict(self, X):
        if not isinstance(X, torch.Tensor):
            raise TypeError("X повинен бути об'єктом torch.Tensor.")

        if X.device != self.other.device:
            raise RuntimeError(
                f"X знаходиться на {X.device}, а модель на {self.other.device}."
            )

        return X * self.price_per_carat + self.other


class NeuralNetwork2:
    def __init__(self, device="cpu"):
        self.device = device

        self.price_per_carat = torch.tensor(
            [50.],   # початкова ціна в каратах
            requires_grad=True,
            device=device
        )

        self.price_per_carat2 = torch.tensor(
            [10.],   # початкова ціна за карат в квадраті
            requires_grad=True,
            device=device
        )

        self.other = torch.tensor(
            [42.],  # ціна за додаткові витрати
            requires_grad=True,
            device=device
        )

    def predict(self, X):
        if not isinstance(X, torch.Tensor):
            raise TypeError("X повинен бути об'єктом torch.Tensor.")

        if X.device != self.other.device:
            raise RuntimeError(
                f"X знаходиться на {X.device}, а модель на {self.other.device}."
            )

        return X * self.price_per_carat + X**2 * self.price_per_carat2 + self.other


In [69]:
class Teacher:
    def __init__(self, model, lr=0.1):
        """
        Створює вчителя для навчання моделі.

        Параметри:
            model (NeuralNetwork): модель, яку потрібно навчати.
            lr (float): швидкість навчання.
        """
        self.model = model

        params = [model.price_per_carat, model.other]
        if hasattr(model, "price_per_carat2"):
            params.append(model.price_per_carat2)
            
        self.optimizer = torch.optim.Adam(
            params,
            lr=lr
        )

    def teach(self, loss):
        """
        Виконує один крок навчання.

        Параметри:
            loss (torch.Tensor): значення функції втрат.
        """
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

In [70]:
# імпорт модуля
import torch


carat = df['carat'].to_numpy()
price = df['price'].to_numpy()

# вага в каратах
X = torch.tensor(carat, dtype=torch.float32)

# справжня ціна
y_true = torch.tensor(price, dtype=torch.float32)


In [77]:
# Ваш код

